In [1]:
import pandas as pd
import numpy as np
import json
import re 
import sys
import itertools

In [2]:
tracks = pd.read_csv('./tracks.csv')

FileNotFoundError: [Errno 2] No such file or directory: './tracks.csv'

In [3]:
artists = pd.read_csv('./artists.csv')
artists.head()

FileNotFoundError: [Errno 2] No such file or directory: './artists.csv'

In [4]:
tracks['id_artists'] = [i[2:-2] for i in tracks['id_artists']]
tracks['release_year'] = [int(i.split('-')[0]) for i in tracks['release_date']]



tracks.head()

,id,name,popularity,duration_ms,explicit,artists,id_artists,release_date,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,release_year
0,35iwgR4jXetI318WEWsa1Q,Carve,6,126903,0,['Uli'],45tIt06XoI0Iio4LBEVpls,1922-02-22,0.645,0.4450,...,-13.338,1,0.4510,0.674,0.7440,0.151,0.127,104.851,3,1922
1,021ht4sdgPcrDgSk7JTbKY,Capítulo 2.16 - Banquero Anarquista,0,98200,0,['Fernando Pessoa'],14jtPCOoNZwquk5wd9DxrY,1922-06-01,0.695,0.2630,...,-22.136,1,0.9570,0.797,0.0000,0.148,0.655,102.009,1,1922
2,07A5yehtSnoedViJAZkNnc,Vivo para Quererte - Remasterizado,0,181640,0,['Ignacio Corsini'],5LiOoJbxVSAMkBS2fUm3X2,1922-03-21,0.434,0.1770,...,-21.180,1,0.0512,0.994,0.0218,0.212,0.457,130.418,5,1922
3,08FmqUhxtyLTn6pAh6bk45,El Prisionero - Remasterizado,0,176907,0,['Ignacio Corsini'],5LiOoJbxVSAMkBS2fUm3X2,1922-03-21,0.321,0.0946,...,-27.961,1,0.0504,0.995,0.9180,0.104,0.397,169.980,3,1922
4,08y9GfoqCWfOGsKdwojr5e,Lady of the Evening,0,163080,0,['Dick Haymes'],3BiJGZsyX9sJchTqcSA7Su,1922,0.402,0.1580,...,-16.900,0,0.0390,0.989,0.1300,0.311,0.196,103.220,4,1922


In [5]:
artists.rename(columns = {'id': 'id_artists','popularity': 'artists_popularity'}, inplace = True)

artists

,id_artists,followers,genres,name,artists_popularity
0,0DheY5irMjBUeLybbCUEZ2,0.0,[],Armid & Amir Zare Pashai feat. Sara Rouzbehani,0
1,0DlhY15l3wsrnlfGio2bjU,5.0,[],ปูนา ภาวิณี,0
2,0DmRESX2JknGPQyO15yxg7,0.0,[],Sadaa,0
3,0DmhnbHjm1qw6NCYPeZNgJ,0.0,[],Tra'gruda,0
4,0Dn11fWM7vHQ3rinvWEl4E,2.0,[],Ioannis Panoutsopoulos,0
...,...,...,...,...,...
1162090,3cOzi726Iav1toV2LRVEjp,4831.0,['black comedy'],Ali Siddiq,34
1162091,6LogY6VMM3jgAE6fPzXeMl,46.0,[],Rodney Laney,2
1162092,19boQkDEIay9GaVAWkUhTa,257.0,[],Blake Wexler,10
1162093,5nvjpU3Y7L6Hpe54QuvDjy,2357.0,['black comedy'],Donnell Rawlings,15


In [6]:
artists.dtypes
#remove artists with songs before 2010

id_artists             object
followers             float64
genres                 object
name                   object
artists_popularity      int64
dtype: object

In [7]:
tracks.dtypes

id                   object
name                 object
popularity            int64
duration_ms           int64
explicit              int64
artists              object
id_artists           object
release_date         object
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
release_year          int64
dtype: object

In [8]:
tracks['release_date'] = pd.to_datetime(tracks['release_date'], format = '%Y-%m-%d')
tracks = tracks[tracks['release_year']>=2010]
tracks = tracks[tracks['speechiness']<0.66] #above 0.66 are tracks with only spoken words, not music
tracks.shape

(125058, 21)

In [9]:
tracks

,id,name,popularity,duration_ms,explicit,artists,id_artists,release_date,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,release_year
39511,6Pkt6qVikqPBt9bEQy8iTz,A Lover's Concerto,41,159560,0,['The Toys'],6lH5PpuiMa5SpfjoIOlwCS,2020-03-13,0.671,0.867,...,-2.706,1,0.0571,0.436,0.000000,0.1390,0.8390,120.689,4,2020
39529,1hx7X9cMXHWJjknb9O6Ava,The September Of My Years - Live At The Sands ...,26,187333,0,['Frank Sinatra'],1Mxqyy3pSjf8kZZL4QVxS0,2018-05-04,0.319,0.201,...,-17.796,1,0.0623,0.887,0.000000,0.9040,0.2390,117.153,3,2018
39533,19oquvXf3bc65GSqtPYA5S,It Was A Very Good Year - Live At The Sands Ho...,25,236800,0,['Frank Sinatra'],1Mxqyy3pSjf8kZZL4QVxS0,2018-05-04,0.269,0.129,...,-18.168,0,0.0576,0.938,0.000005,0.6830,0.1600,82.332,3,2018
39581,55qyghODi24yaDgKBI6lx0,"The Circle Game - Live at The 2nd Fret, Philad...",18,313093,0,['Joni Mitchell'],5hW4L92KnC6dX9t7tYM4Ve,2020-10-30,0.644,0.212,...,-14.118,1,0.0347,0.881,0.000022,0.7980,0.4410,117.072,3,2020
39583,00xemFYjQNRpOlPhVaLAHa,"Urge For Going - Live at The 2nd Fret, Philade...",18,295093,0,['Joni Mitchell'],5hW4L92KnC6dX9t7tYM4Ve,2020-10-30,0.627,0.184,...,-15.533,1,0.0450,0.955,0.000162,0.0986,0.2990,115.864,4,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
586667,5rgu12WBIHQtvej2MdHSH0,云与海,50,258267,0,['阿YueYue'],1QLBXKM5GCpyQQSVMNZqrZ,2020-09-26,0.560,0.518,...,-7.471,0,0.0292,0.785,0.000000,0.0648,0.2110,131.896,4,2020
586668,0NuWgxEp51CutD2pJoF4OM,blind,72,153293,0,['ROLE MODEL'],1dy5WNgIKQU6ezkpZs4y8z,2020-10-21,0.765,0.663,...,-5.223,1,0.0652,0.141,0.000297,0.0924,0.6860,150.091,4,2020
586669,27Y1N4Q4U3EfDU5Ubw8ws2,What They'll Say About Us,70,187601,0,['FINNEAS'],37M5pPGs6V1fchFJSgCguX,2020-09-02,0.535,0.314,...,-12.823,0,0.0408,0.895,0.000150,0.0874,0.0663,145.095,4,2020
586670,45XJsGpFTyzbzeWK8VzR8S,A Day At A Time,58,142003,0,"['Gentle Bones', 'Clara Benin']","4jGPdu95icCKVF31CcFKbS', '5ebPSE9YI5aLeZ1Z2gkqjn",2021-03-05,0.696,0.615,...,-6.212,1,0.0345,0.206,0.000003,0.3050,0.4380,90.029,4,2021


In [10]:
tracks['artists'] = [i[2:-2] for i in tracks['artists']]

tracks[tracks['name'] == 'Adore You']

,id,name,popularity,duration_ms,explicit,artists,id_artists,release_date,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,release_year
86217,5AnCLGg35ziFOloEnXK4uu,Adore You,71,278747,0,Miley Cyrus,5YGY8feqx7naU7z4HrwZM6,2013-10-04,0.583,0.655,...,-5.407,1,0.0315,0.1110,0.000004,0.113,0.201,119.759,4,2013
91884,3jjujdWJ72nww5eGnfs2E7,Adore You,88,207133,0,Harry Styles,6KImCVD70vtIoJWnq6nGn3,2019-12-13,0.676,0.771,...,-3.675,1,0.0483,0.0237,0.000007,0.102,0.569,99.048,4,2019
92524,1M4qEo4HE3PRaCOM7EXNJq,Adore You,74,207133,0,Harry Styles,6KImCVD70vtIoJWnq6nGn3,2019-12-06,0.676,0.771,...,-3.675,1,0.0483,0.0237,0.000007,0.102,0.569,99.048,4,2019


In [11]:
artists[artists['name'] == "Harry Styles"]

,id_artists,followers,genres,name,artists_popularity
115494,6KImCVD70vtIoJWnq6nGn3,14086781.0,"['pop', 'post-teen pop']",Harry Styles,90


In [12]:
tracks.to_csv('tracksProcessed.csv')
artists.to_csv('artistsProcessed.csv')